In [6]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "verl").is_dir() else CWD.parent
TOY_DIR = ROOT / "tests" / "special_standalone" / "future_returns_variance"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(TOY_DIR))

from variance_study import (  # noqa: E402
    divergence_name,
    future_returns,
    remaining_weights,
    token_divergence,
)

LETTERS = tuple("abcdefghijklmnopqrstuvwxyz")
N_LETTERS = len(LETTERS)


class CausalBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True, bias=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        t = x.shape[1]
        causal = torch.ones(t, t, device=x.device, dtype=torch.bool).triu(1)
        attn, _ = self.attn(h, h, h, attn_mask=causal, need_weights=False)
        x = x + attn
        h = self.ln2(x)
        return x + self.fc2(F.gelu(self.fc1(h)))


class TinyTransformer(nn.Module):
    def __init__(self, n_letters: int, d_model: int, n_layers: int, n_heads: int, d_ff: int, max_len: int):
        super().__init__()
        self.embed = nn.Embedding(n_letters + 1, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList(CausalBlock(d_model, n_heads, d_ff) for _ in range(n_layers))
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_letters, bias=False)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        pos = torch.arange(idx.shape[1], device=idx.device)
        x = self.embed(idx) + self.pos(pos)
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))
    
    
@torch.no_grad()
def rollout(model: nn.Module, n_seq: int, max_respose_len: int, n_rollouts:int, input_len:int) -> torch.Tensor:
    device = next(model.parameters()).device
    all_responses = []
    response_mask = torch.zeros(n_rollouts, n_seq, max_respose_len+input_len, device=device)
    prompts = torch.randint(0, N_LETTERS, (n_seq, input_len), device=device)
    for rollout_idx in range(n_rollouts):
        start_letters = prompts
        for _ in range(max_respose_len):
                logits = model(start_letters)[:, -1]
                letter = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
                start_letters = torch.cat([start_letters, letter], dim=1)
        response_mask[rollout_idx, :, -max_respose_len:] = 1
        all_responses.append(start_letters)
    all_responses = torch.stack(all_responses).reshape(-1, max_respose_len+input_len)
    response_mask = response_mask.reshape(-1, max_respose_len+input_len)
    return all_responses, response_mask

def flat_grad(grads) -> torch.Tensor:
    return torch.cat([g.reshape(-1) for g in grads])

def true_gradient(student, teacher, sequences, response_mask, alpha, input_len):
    divergence = lambda s, t: token_divergence(s.log(), t, t.log(), alpha)
    student_logits = student(sequences)
    teacher_logits = teacher(sequences).detach()
    student_probs = F.softmax(student_logits, dim=-1)[:, input_len-1:-1]
    teacher_probs = F.softmax(teacher_logits, dim=-1).detach()[:, input_len-1:-1]

    trajectory_log_prob = student_probs.log().gather(dim=-1, index = sequences[:, input_len:].unsqueeze(-1)).squeeze(-1).sum(dim=1)
    # set idx just before response beginning to 1
    loss = divergence(student_probs, teacher_probs).sum(-1)
    loss = loss * torch.exp(trajectory_log_prob - trajectory_log_prob.detach())
    student.zero_grad()
    loss.mean().backward()
    return flat_grad([p.grad for p in student.parameters() if p.requires_grad])



In [7]:
# Loss formulations ported from notebook/test2.ipynb. Sequences are [prompt | response];
# logits at position j predict token j+1, so positions input_len-1 .. L-2 score the response.


def init_model(seed: int) -> TinyTransformer:
    g = torch.Generator().manual_seed(seed)
    model = TinyTransformer(N_LETTERS, D_MODEL, N_LAYERS, N_HEADS, D_FF, max_len=INPUT_LEN + T)
    for p in model.parameters():
        if p.dim() >= 2:
            nn.init.normal_(p, mean=0.0, std=INIT_STD, generator=g)
    return model


def student_params(model: nn.Module) -> list[nn.Parameter]:
    return [p for p in model.parameters() if p.requires_grad]


def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    return F.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()


def sequence_rewards(student, teacher, sequences, alpha: float, input_len: int):
    s_logp = F.log_softmax(student(sequences), dim=-1)[:, input_len - 1:-1]
    with torch.no_grad():
        t_logp = F.log_softmax(teacher(sequences), dim=-1)[:, input_len - 1:-1]
    rewards = token_divergence(s_logp, t_logp.exp(), t_logp, alpha)
    logp_a = s_logp.gather(-1, sequences[:, input_len:].unsqueeze(-1)).squeeze(-1)
    return rewards, logp_a


def batch_grad(student, teacher, sequences, weight_fn, alpha: float, input_len: int) -> torch.Tensor:
    rewards, logp_a = sequence_rewards(student, teacher, sequences, alpha, input_len)
    weight = weight_fn(rewards.detach())
    loss = (rewards.sum(-1) + (weight * logp_a).sum(-1)).mean()
    return flat_grad(torch.autograd.grad(loss, student_params(student)))


def per_sample_grads(student, teacher, sequences, weight_fn, alpha: float, input_len: int) -> torch.Tensor:
    rewards, logp_a = sequence_rewards(student, teacher, sequences, alpha, input_len)
    weight = weight_fn(rewards.detach())
    per_seq = rewards.sum(-1) + (weight * logp_a).sum(-1)
    params = student_params(student)
    rows = [
        flat_grad(torch.autograd.grad(per_seq[i], params, retain_graph=True))
        for i in range(per_seq.shape[0])
    ]
    return torch.stack(rows)


def grad_metrics(g: torch.Tensor, g_star: torch.Tensor) -> dict:
    mean = g.mean(0)
    noise = (g - mean).pow(2).sum(-1).mean().sqrt()
    snr = (mean.norm() / noise).item() if float(noise) > 0 else float("nan")
    return {"snr": snr, "cos": cosine(mean, g_star), "mean_norm": mean.norm().item(), "noise": noise.item()}


def credit_weight_fns(length: int):
    rem = remaining_weights(length)

    def as_rem(g):
        return rem.to(device=g.device, dtype=g.dtype)

    return {
        "raw G_t": lambda r: future_returns(r),
        "G_t - batch_mean_t": lambda r: future_returns(r) - future_returns(r).mean(0, keepdim=True),
        "G(r - batch_mean_t)": lambda r: future_returns(r - r.mean(0, keepdim=True)),
        "G_t / remaining": lambda r: future_returns(r) / as_rem(r),
        "(G_t - batch_mean_t)/remaining": lambda r: (
            future_returns(r) - future_returns(r).mean(0, keepdim=True)
        ) / as_rem(r),
        "G(r - batch_mean_t)/remaining": lambda r: future_returns(r - r.mean(0, keepdim=True)) / as_rem(r),
        "G_t / (T-1)": lambda r: future_returns(r) / (r.shape[1] - 1),
        "G(r - batch_mean_t)/(T-1)": lambda r: future_returns(r - r.mean(0, keepdim=True)) / (r.shape[1] - 1),
        "no PG term": lambda r: torch.zeros_like(r),
    }


def heldout_weight_fn(baseline_r: torch.Tensor):
    baseline_G = future_returns(baseline_r.reshape(1, -1)).reshape(-1)

    def fn(rewards: torch.Tensor) -> torch.Tensor:
        return future_returns(rewards) - baseline_G.to(device=rewards.device, dtype=rewards.dtype)

    return fn


def trajectory_weight_fn(baseline_R: float):
    def fn(rewards: torch.Tensor) -> torch.Tensor:
        return (rewards.sum(-1, keepdim=True) - baseline_R).expand_as(rewards)

    return fn


In [8]:
D_MODEL = 64
N_LAYERS = 4
N_HEADS = 8
D_FF = 64
INPUT_LEN = 8  # random prompt letters
T = 32  # response length
ALPHA = 0.0  # 0=forward KL, 1=reverse KL, 0.5=JSD
SEED = 0
INIT_STD = 0.15
N_PROMPTS = 512  # true gradient: N_PROMPTS prompts x N_COMPLETIONS completions each
N_COMPLETIONS = 16
N_BASE = 256
N_METRICS = 32

torch.manual_seed(SEED)
DEVICE = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
student = init_model(SEED).to(DEVICE)
teacher = init_model(SEED + 1).to(DEVICE).eval()
for p in teacher.parameters():
    p.requires_grad_(False)
student.train()
n_params = sum(p.numel() for p in student.parameters())
print(
    f"device={DEVICE}  divergence={divergence_name(ALPHA)}  depth={N_LAYERS}  "
    f"d_model={D_MODEL}  prompt={INPUT_LEN}  T={T}  params={n_params}"
)


device=cuda:3  divergence=forward_kl  depth=4  d_model=64  prompt=8  T=32  params=105920


In [9]:
# True on-policy gradient: N_PROMPTS x N_COMPLETIONS on-policy samples, one loss over all of them.
true_seqs, true_mask = rollout(student, N_PROMPTS, T, N_COMPLETIONS, INPUT_LEN)
g_true = true_gradient(student, teacher, true_seqs, true_mask, ALPHA, INPUT_LEN).clone()

# Port check: test2's batch_grad with the whole-trajectory weight (b = 0) is the same loss.
g_port = batch_grad(student, teacher, true_seqs, trajectory_weight_fn(0.0), ALPHA, INPUT_LEN)

# Noise check: split the prompts in half (rows are completion-major, so prompt = row % N_PROMPTS).
first = (torch.arange(true_seqs.shape[0], device=true_seqs.device) % N_PROMPTS) < N_PROMPTS // 2
g_a = true_gradient(student, teacher, true_seqs[first], true_mask[first], ALPHA, INPUT_LEN).clone()
g_b = true_gradient(student, teacher, true_seqs[~first], true_mask[~first], ALPHA, INPUT_LEN).clone()

print(f"sequences={true_seqs.shape[0]}  ||g_true||={g_true.norm():.3f}")
print(f"true_gradient vs batch_grad(R, b=0): rel err={((g_true - g_port).norm() / g_true.norm()).item():.2e}")
print(f"split-half cosine (by prompt)={cosine(g_a, g_b):.3f}")


sequences=8192  ||g_true||=103.406
true_gradient vs batch_grad(R, b=0): rel err=4.29e-07
split-half cosine (by prompt)=0.526


In [10]:
# test2 credit weights, each the mean of N_METRICS per-sequence grads, against g_true.
baseline_seqs, _ = rollout(student, N_BASE, T, 1, INPUT_LEN)
with torch.no_grad():
    rewards_base, _ = sequence_rewards(student, teacher, baseline_seqs, ALPHA, INPUT_LEN)
baseline_r = rewards_base.mean(0)
rtg_w = heldout_weight_fn(baseline_r)
traj_w = trajectory_weight_fn(rewards_base.sum(-1).mean().item())

weight_fns = {
    "full trajectory (R)": trajectory_weight_fn(0.0),
    "full trajectory (R - b)": traj_w,
    "reward-to-go held-out baseline": rtg_w,
    **credit_weight_fns(T),
}
metric_seqs, _ = rollout(student, N_METRICS, T, 1, INPUT_LEN)
rows = []
for name, fn in weight_fns.items():
    g = per_sample_grads(student, teacher, metric_seqs, fn, ALPHA, INPUT_LEN)
    rows.append({"variant": name, **grad_metrics(g, g_true)})
credit_df = pd.DataFrame(rows)
display(credit_df.round(4))


,variant,snr,cos,mean_norm,noise
0,full trajectory (R),0.1670,0.0534,826.4933,4949.8140
1,full trajectory (R - b),0.1904,0.3206,199.4944,1047.8696
2,reward-to-go held-out baseline,0.1992,0.3809,132.0433,662.7833
3,raw G_t,0.1729,0.0463,483.9498,2798.5603
4,G_t - batch_mean_t,0.1983,0.3792,131.4232,662.5900
5,G(r - batch_mean_t),0.1983,0.3792,131.4232,662.5901
6,G_t / remaining,0.4015,0.7455,79.0462,196.8820
7,(G_t - batch_mean_t)/remaining,0.5525,0.7627,78.7150,142.4714
8,G(r - batch_mean_t)/remaining,0.5525,0.7627,78.7150,142.4714
9,G_t / (T-1),0.4920,0.7565,77.9488,158.4203
